# High level RSA encryption

## Introduction
This file gives you a quick introduction to the RSA algorithm, before presenting a high level Python implementation of RSA. The implementation is created in such a way that it should be easily transferable to a digital design (e.g. no np.mod operations).

## RSA crash course
RSA is an encryption algorithm that relies on the simple fact that factoring prime numbers is a hard problem. The algorithms are the following:

$Enc(M, pk, n) := C = M^{pk} (mod \ n) \newline$
$Dec(C, sk, n) := M = C^{d} (mod \ n) \newline$

Where M - message, C - ciphertext, pk - public key, sk - secret key, n - modulus. Some important notes is that pk, sk and n is given to us in this task. n is always a product of two large prime numbers.

In [ ]:
"""
RSA prototype

Using the two simplest / most parallel-friendly algorithms from Koc, "RSA Hardware Implementation"
(Bell Labs, 1995):

    modexp  -> RL binary method            (Sec. 4)   -- independent square/multiply steps
    modmul  -> Interleaving shift-add-reduce (Sec. 7.1) -- the simplest modmul in the report
    add     -> bit-level carry-propagate adder (Sec. 5.2)

Bit vectors (LSB-first lists of 0/1) are used for every arithmetic step in
the datapath, so this maps directly onto an RTL structure later.
"""

from typing import List, Tuple

BitVec = List[int]  # LSB-first bit vector, e.g. 0b1010 -> [0,1,0,1]


# ---------------------------------------------------------------------------
# Level 0: bit-vector <-> int helpers (testbench convenience only --
# not part of the hardware datapath itself)
# ---------------------------------------------------------------------------

def to_bits(x: int, width: int) -> BitVec:
    return [(x >> i) & 1 for i in range(width)]


def from_bits(bits: BitVec) -> int:
    return sum(b << i for i, b in enumerate(bits))


# ---------------------------------------------------------------------------
# Level 1: Addition Operation (Section 5)
#
# A k-bit Carry Propagate Adder (CPA): a linear ripple of full-adder cells.
# Simplest adder in the report (O(k) delay) -- swap this out for a
# carry-save/carry-look-ahead model later without touching any layer above.
# ---------------------------------------------------------------------------

def full_adder(a: int, b: int, cin: int) -> Tuple[int, int]:
    """One FA cell. Returns (sum_bit, carry_out)."""
    s = a ^ b ^ cin
    cout = (a & b) | (a & cin) | (b & cin)
    return s, cout


def cpa_add(a_bits: BitVec, b_bits: BitVec, width: int) -> Tuple[BitVec, int]:
    """k-bit ripple-carry addition. Returns (sum_bits, carry_out)."""
    sum_bits = [0] * width
    carry = 0
    for i in range(width):
        sum_bits[i], carry = full_adder(a_bits[i], b_bits[i], carry)
    return sum_bits, carry


def cpa_sub(a_bits: BitVec, b_bits: BitVec, width: int) -> Tuple[BitVec, int]:
    """a - b via 2's-complement addition (Sec. 5.2).
    carry_out = 1 means a >= b (no borrow); carry_out = 0 means a < b."""
    b_inv = [1 - bit for bit in b_bits]
    sum_bits = [0] * width
    carry = 1  # inject +1 for 2's complement
    for i in range(width):
        sum_bits[i], carry = full_adder(a_bits[i], b_inv[i], carry)
    return sum_bits, carry


# ---------------------------------------------------------------------------
# Level 2: Modular Multiplication -- Interleaving method (Section 7.1)
#
# This is the simplest modular multiplier in the report: shift, add, reduce,
# one bit of B at a time.
#
#   P := 0
#   for i = 0 to k-1:
#       P := 2P + A * B_{k-1-i}
#       P := P mod n            (at most 2 subtractions of n, see below)
#
# Why at most 2 subtractions: with A, B, P < n, each step gives
#   P <= 2(n-1) + (n-1) = 3n - 3,
# so P lands in [0, 3n-3] and at most two conditional subtractions of n
# bring it back into [0, n).
#
# The left shift (2P) is pure wiring in hardware -- no adder needed. The
# partial product A*B_j is a single AND-gate array (B_j is one bit). Only
# the accumulation and the reduction actually use the Level-1 adder.
# ---------------------------------------------------------------------------

def mod_mul(a: int, b: int, n: int, k: int = None) -> int:
    """P = A*B mod n via the interleaving shift-add-reduce method."""
    if k is None:
        k = n.bit_length()
    width = k + 2  # room for P up to 3n - 3

    b_bits = to_bits(b, k)
    n_bits = to_bits(n, width)
    P = 0

    for i in range(k):
        bit = b_bits[k - 1 - i]          # scan B from MSB to LSB
        P_shifted = 2 * P                # step: left shift (wiring)
        partial = a if bit else 0        # step: partial product (AND array)

        p_bits = to_bits(P_shifted, width)
        partial_bits = to_bits(partial, width)
        sum_bits, _ = cpa_add(p_bits, partial_bits, width)
        P = from_bits(sum_bits)          # step: P := 2P + A*B_j

        for _ in range(2):               # step: at most 2 subtractions
            p_bits = to_bits(P, width)
            diff_bits, no_borrow = cpa_sub(p_bits, n_bits, width)
            if no_borrow:
                P = from_bits(diff_bits)

    return P


# ---------------------------------------------------------------------------
# Level 3: Modular Exponentiation -- RL Binary Method (Section 4)
#
#   C := 1 ; P := M
#   for i = 0 to h-2:
#       if e_i == 1: C := C*P mod n     -- step 2a
#       P := P*P mod n                  -- step 2b
#   if e_{h-1} == 1: C := C*P mod n
#   return C
#
# The report calls out explicitly that steps 2a and 2b in each iteration
# depend only on the *previous* C and P, not on each other -- so with two
# multipliers available (one for the conditional multiply, one for the
# squaring) they can run in parallel each cycle. That is the whole reason
# to prefer RL over LR here, at the cost of a third register (P, alongside
# M and C -- though P can reuse M's register once M is no longer needed).
# ---------------------------------------------------------------------------

def mod_exp(M: int, e: int, n: int, k: int = None) -> int:
    """C = M^e mod n, RL binary exponentiation over the interleaving multiplier."""
    if k is None:
        k = n.bit_length()

    h = e.bit_length()
    if h == 0:
        return 1 % n

    e_bits = to_bits(e, h)  # LSB-first
    C = 1
    P = M % n

    for i in range(h - 1):
        if e_bits[i]:
            C = mod_mul(C, P, n, k)   # step 2a -- independent of step 2b
            # (in hardware: dispatch this multiply and the squaring below
            #  to two separate multiplier units in the same cycle)
        P = mod_mul(P, P, n, k)       # step 2b

    if e_bits[h - 1]:
        C = mod_mul(C, P, n, k)       # step 3

    return C


# ---------------------------------------------------------------------------
# Self-test
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Worked example straight from the report, Section 1:
    # n = 143 = 11*13, e = 17, d = 113, M = 50 -> C = 85 -> M = 50
    n, e, d, M = 143, 17, 113, 50
    C = mod_exp(M, e, n)
    M_recovered = mod_exp(C, d, n)
    print(f"n={n}, e={e}, d={d}, M={M}")
    print(f"Encrypt: C = M^e mod n = {C}   (expected 85)")
    print(f"Decrypt: M = C^d mod n = {M_recovered}   (expected {M})")
    assert C == 85
    assert M_recovered == M

    # Same M^55 example the report walks through for the RL method (Sec. 4)
    assert mod_exp(2, 55, 1_000_000_007) == pow(2, 55, 1_000_000_007)

    # Larger modulus, cross-checked against Python's built-in pow()
    n2 = 0xC4F8E1B7A3D2F5091E7B4A6C8D3F1A2B5E9C7D4F6A8B1E3D5F7A9C2B4E6D8F01
    e2 = 65537
    M2 = 123456789
    C2 = mod_exp(M2, e2, n2)
    C2_ref = pow(M2, e2, n2)
    print(f"\nLarge modulus check: C2 == pow(M2,e2,n2)? {C2 == C2_ref}")
    assert C2 == C2_ref

    print("\nAll checks passed.")

n=143, e=17, d=113, M=50
Encrypt: C = M^e mod n = 85   (expected 85)
Decrypt: M = C^d mod n = 50   (expected 50)

Large modulus check: C2 == pow(M2,e2,n2)? True

All checks passed.
